# Project 5: E-commerce Customer Behavior & Segmentation
Segment customers and predict purchase patterns to build a recommendation and retention system.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("datasets/ecommerce_customer_data.csv")
print("Dataset Loaded. Shape:", df.shape)
df.head()

Dataset Loaded. Shape: (12000, 14)


,CustomerID,Gender,Age,Location,SignupDate,LastPurchaseDate,OrderCount,TotalSpend,FavCategory,AverageRating,Recency,Frequency,Monetary,ChurnRisk
0,CUST-00001,Female,59,Metro,2024-06-14,2024-08-24,3,2478.60,Beauty,3.5,495,3,2478.60,0
1,CUST-00002,Male,58,Urban,2024-09-22,2025-07-22,11,167072.33,Electronics,3.9,163,11,167072.33,0
2,CUST-00003,Female,40,Suburban,2024-04-13,2024-06-10,28,28860.01,Books,3.9,570,28,28860.01,0
3,CUST-00004,Female,37,Urban,2024-04-26,2025-09-25,37,34701.19,Books,3.4,98,37,34701.19,0
4,CUST-00005,Female,61,Metro,2023-02-18,2024-08-07,5,6400.18,Beauty,4.3,512,5,6400.18,0


## 1. RFM Clustering (K-Means)
Segment customers into distinct value groups using K-Means clustering on Recency, Frequency, and Monetary metrics.

In [2]:
# Normalize RFM fields
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df[['Recency', 'Frequency', 'Monetary']])

# Run K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Plot Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Recency', y='Monetary', hue='Cluster', size='Frequency', palette='tab10', sizes=(20, 200), alpha=0.6)
plt.title("E-commerce Customer Clusters (RFM)")
plt.xlabel("Recency (Days since last purchase)")
plt.ylabel("Monetary Spend (₹)")
plt.yscale('log')
plt.tight_layout()
plt.savefig("visualizations/customer_clusters.png")
plt.savefig("../visualizations/customer_clusters.png")
plt.show()

print("Cluster Profiles (Mean values):")
print(df.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean())

Cluster Profiles (Mean values):
            Recency  Frequency       Monetary
Cluster                                      
0        208.796228  12.650703   45549.353510
1        368.618433  27.494931  417397.225318
2        649.070905  14.557867   52013.224360
3        309.220153  33.785100   70014.974250


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_2828\1703007619.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Customer Churn Risk Analysis
We profile the 320 high-risk churn customers identified based on purchase recency, purchase count, and satisfaction rating.

In [3]:
# Check total churn risk
print("Total High Churn Risk Customers:", df['ChurnRisk'].sum())

# Factors of Churn
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='ChurnRisk', y='Recency', hue='ChurnRisk', palette='Set1', legend=False)
plt.title("Purchase Recency by Churn Risk status")
plt.xticks([0, 1], ["Low Risk", "High Churn Risk"])
plt.ylabel("Recency (Days)")
plt.tight_layout()
plt.savefig("visualizations/churn_risk_factors.png")
plt.savefig("../visualizations/churn_risk_factors.png")
plt.show()

Total High Churn Risk Customers: 320


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_2828\2017655112.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Customer Lifetime Value (CLV) by Category
Calculate CLV (proxied by total spend) across product categories.

In [4]:
cat_clv = df.groupby("FavCategory")["TotalSpend"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=cat_clv, x='FavCategory', y='TotalSpend', hue='FavCategory', palette='Set2', legend=False)
plt.title("Average Customer Lifetime Spend by Favorite Category")
plt.ylabel("Average Total Spend (₹)")
plt.tight_layout()
plt.savefig("visualizations/clv_by_category.png")
plt.savefig("../visualizations/clv_by_category.png")
plt.show()

print(cat_clv)

   FavCategory     TotalSpend
0      Apparel   36835.141752
1       Beauty   27236.550353
2        Books   14522.365053
3  Electronics  271869.764990
4   Home Decor   72576.695376


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_2828\2104398910.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Churn Risk Predictive Model & Feature Importance
Train a Random Forest classifier to identify churn risk and evaluate feature importances.

In [5]:
# Preprocessing
X = df[['Age', 'Recency', 'Frequency', 'Monetary', 'AverageRating']]
y = df['ChurnRisk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Plot Feature Importance
importances = model.feature_importances_
features = X.columns
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='Importance', y='Feature', hue='Feature', palette='magma', legend=False)
plt.title("Feature Importance for Churn Prediction Model")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("visualizations/feature_importance.png")
plt.savefig("../visualizations/feature_importance.png")
plt.show()

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3504
           1       0.94      0.77      0.85        96

    accuracy                           0.99      3600
   macro avg       0.97      0.88      0.92      3600
weighted avg       0.99      0.99      0.99      3600



C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_2828\1636502983.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
